In [6]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import RFE  
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE

## ------------------------------------------------------------------
## STEP 1: Get the data
## ------------------------------------------------------------------
data = yf.download('SPY', start='2015-01-01', end='2025-1-1')

## ------------------------------------------------------------------
## STEP 2: Create Features
## ------------------------------------------------------------------
# If the columns are a MultiIndex, flatten them
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# Add technical indicators using pandas_ta
data.ta.ema(length=100, append=True)  # Exponential Moving Average
data.ta.sma(length=25, append=True)   # Simple Moving Average
data.ta.rsi(length=10, append=True)  # Relative Strength Index
data.ta.vwap(high='High', low='Low', close='Close', volume='Volume', append=True)  # Volume Weighted Average Price
data.ta.bbands(append=True)  # Bollinger Bands
data.ta.atr(length=14, append=True)  # Average True Range

# print column names to verify
print("--- Actual Column Names ---")
print(data.columns)
print("----------------------------")

# --- NEW "RELATIVE" FEATURES ---
# 1. Price vs. its main moving average (Is price overextended?)
data['Price_vs_SMA25'] = (data['Close'] / data['SMA_25']) - 1

# 2. Fast average vs. Slow average (What's the trend momentum?)
data['SMA25_vs_EMA100'] = (data['SMA_25'] / data['EMA_100']) - 1

# 3. How far is RSI from its 'neutral' 50-mark?
data['RSI_vs_50'] = data['RSI_10'] - 50

# 4. Volatility as a percentage of price (Is volatility high *for this price*?)
data['ATR_Percent'] = (data['ATRr_14'] / data['Close']) * 100


## ------------------------------------------------------------------
## STEP 3: Create Label
## ------------------------------------------------------------------
future_return = data['Close'].pct_change(5).shift(-5)
UPPER_THRESHOLD = 0.02
LOWER_THRESHOLD = -0.02
conditions = [(future_return > UPPER_THRESHOLD), (future_return < LOWER_THRESHOLD)]
choices = [1, -1]
data['Target'] = np.select(conditions, choices, default=0)

## ------------------------------------------------------------------
## STEP 4: Prepare Data for the Model (Same)
## ------------------------------------------------------------------
data = data.dropna()
feature_list = [
    'EMA_100', 'SMA_25', 'RSI_10', 'VWAP_D', 'ATRr_14', # Original group
    'BBL_5_2.0_2.0', 'BBB_5_2.0_2.0', 'BBU_5_2.0_2.0', 'BBP_5_2.0_2.0', # BBands group
    
    # New Contextual Features
    'Price_vs_SMA25', 
    'SMA25_vs_EMA100', 
    'RSI_vs_50',
    'ATR_Percent'
]
X = data[feature_list]
Y = data['Target']

# split the data into training and testing sets (80% train, 20% test)
split_percentage = 0.8
split_index = int(len(data) * split_percentage)
X_train = X[:split_index]
Y_train = Y[:split_index]
X_test = X[split_index:]
Y_test = Y[split_index:]

print("--- Original Class Distribution (Training Set) ---")
print(Y_train.value_counts(normalize=True))


## ------------------------------------------------------------------
## STEP 5: Apply SMOTE to fix Imbalance
## ------------------------------------------------------------------
smote = SMOTE(random_state=42) 

# Fit and apply SMOTE ONLY to the training data
X_train_resampled, Y_train_resampled = smote.fit_resample(X_train, Y_train)

print("--- Class Distribution (After SMOTE) ---")
print(Y_train_resampled.value_counts(normalize=True))

## ------------------------------------------------------------------
## STEP 6: Train and Evaluate the FINAL Model (No RFE)
## ------------------------------------------------------------------
param_grid = {
    'n_estimators': [200, 500],
    'max_depth': [3, 5, 10],
    'min_samples_leaf': [5, 10, 20],
    'max_features': ['sqrt', 0.5]
}

# Create the grid search object
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,                 # 3-fold cross-validation
    scoring='f1_macro',   # Optimize for 'f1_macro'
    n_jobs=-1             # Use all CPU cores
)

# Fit the grid search on the *full resampled* training data
print(f"\nRunning GridSearchCV on all {len(feature_list)} features...")
grid_search.fit(X_train_resampled, Y_train_resampled)

# See the best settings it found
print(f"Best parameters found: {grid_search.best_params_}")

# The grid_search object is now your best model
# Predict on the *original, non-resampled* test set (X_test)
predictions = grid_search.predict(X_test)

# Now check the report
print("\n--- Final Classification Report ---")
print(classification_report(Y_test, predictions, target_names=['Sell (-1)', 'Hold (0)', 'Buy (1)']))


/tmp/ipykernel_778/2132479205.py:15: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('SPY', start='2015-01-01', end='2025-1-1')
[*********************100%***********************]  1 of 1 completed


--- Actual Column Names ---
Index(['Close', 'High', 'Low', 'Open', 'Volume', 'EMA_100', 'SMA_25', 'RSI_10',
       'VWAP_D', 'BBL_5_2.0_2.0', 'BBM_5_2.0_2.0', 'BBU_5_2.0_2.0',
       'BBB_5_2.0_2.0', 'BBP_5_2.0_2.0', 'ATRr_14'],
      dtype='object', name='Price')
----------------------------
--- Original Class Distribution (Training Set) ---
Target
 0    0.718572
 1    0.154682
-1    0.126746
Name: proportion, dtype: float64
--- Class Distribution (After SMOTE) ---
Target
 0    0.333333
-1    0.333333
 1    0.333333
Name: proportion, dtype: float64

Running GridSearchCV on all 13 features...
Best parameters found: {'max_depth': 10, 'max_features': 0.5, 'min_samples_leaf': 5, 'n_estimators': 500}

--- Final Classification Report ---
              precision    recall  f1-score   support

   Sell (-1)       0.10      0.81      0.18        42
    Hold (0)       0.87      0.27      0.42       363
     Buy (1)       0.37      0.19      0.25        79

    accuracy                           